### Workflow for K-fold CV

In [ ]:
import pandas as pd
import numpy as np, networkx as nx, gudhi_persistence as gp, watts_model as nwm
import matplotlib.pyplot as plt, utilsA1 as utils
import os, importlib

from sklearn.decomposition import PCA
from lifelines import CoxTimeVaryingFitter


In [ ]:

num_samples = 10

param_generation_rng = np.random.default_rng(444)
params_list = []
for _ in range(num_samples):
    num_nodes = param_generation_rng.choice([20])
    weighted = True
    n_seeds = param_generation_rng.choice([2])
    node_active_threshold = param_generation_rng.choice([0.02, 0.025, 0.03, 0.1])
    num_neighbor_nodes = param_generation_rng.choice([3, 4])
    distance_threshold = param_generation_rng.integers(num_neighbor_nodes + 1, num_neighbor_nodes + 2)
    total_random_edges = param_generation_rng.choice([20, 30, 40])
    upper_weight_limit = param_generation_rng.choice([10, 20])
    skew_power = param_generation_rng.choice([2, 4])
    seed_cluster_distance = 20
    ngeom_edges_in_persistence = False
    max_persistence_dim = 2
    threshold_sum = sum(range(num_nodes))
    seeding_method = 'all_combinations'
    ngeo_placement = 'random.choice'
    calculate_representation = True

    param = {
        'num_nodes': num_nodes,  # fixed
        'num_neighbor_nodes': num_neighbor_nodes,
        'total_random_edges': total_random_edges,
        'distance_threshold': distance_threshold,
        'weighted': weighted,
        'ngeo_placement': ngeo_placement,  # other 'ngeo_per_node'
        'n_seeds': n_seeds,
        'node_active_threshold': node_active_threshold,
        'upper_weight_limit': upper_weight_limit,
        'skew_power': skew_power,
        'seed_cluster_distance': seed_cluster_distance,
        'ngeom_edges_in_persistence': ngeom_edges_in_persistence,
        'max_persistence_dim': max_persistence_dim,
        'threshold_sum': threshold_sum,
        'seeding_method': seeding_method,
        'calculate_representation': calculate_representation,
        'bandwidth': 0.4
    }

    params_list.append(param)

param_df = pd.DataFrame(params_list)
param_df


In [ ]:
front_cols = ["simulation_id", "realization_id", "time", "state",
              "node_active_threshold", "num_nodes", "num_neighbor_nodes",
              "total_random_edges", "num_active_nodes", "active_nodes"]
base_cols = ['simulation_id', 'realization_id', 'time', 'state', 'E_0', 'E_1', 'E_2']
feature_cols = ['L_0', 'L_1', 'L_2', 'I_0', 'I_1', 'I_2']
bandwidth_grid = [0.01, 0.02, 0.05, 0.1, 0.2, 0.25, 0.5, 0.8, 1.0, 2.0]
representation_choice_function_grid = ['persistence', 'birth', 'arctan']


In [ ]:
validation_results = []

for bandwidth in bandwidth_grid:
    for surface_fun in representation_choice_function_grid:
        print(f"Simulation and validation for: surface_fun={surface_fun}, bandwidth={bandwidth}")
        params_list[0]['representation_choice_function'] = surface_fun
        params_list[0]['bandwidth'] = bandwidth
        df, _ = nwm.main_sims(params_list = params_list, save_files = False)
        df, _ = utils.clean_simulation_df(df,
                                        group_cols=['simulation_id', 'realization_id'],
                                        front_cols=front_cols,
                                        suffix_prefixes=['H', 'L', 'I', 'E'])
        df = utils.compute_pca_features(df, feature_cols, base_cols, n_components=5)
        df = utils.prep_for_cox_tv(df,
                                         group_cols=['simulation_id', 'realization_id'],
                                         state_col='state',
                                         landscape_prefixes=['L'],
                                         image_prefixes=['I'],
                                         essential_prefixes=['E'])
        cindex_mean = utils.cvt_validation_score(df, k = 5)
        validation_results.append({
            "bandwidth": bandwidth,
            "surface_function": surface_fun,
            "cv_concordance": cindex_mean
        })

In [ ]:

validation_results_df = pd.DataFrame(validation_results)
best = validation_results.loc[validation_results["cv_log_likelihood"].idxmax()]
print("Best hyperparameters:", best)